# Merging tables

In [1]:
from autoencoder.db import get_connection
from autoencoder.db import get_tables
from autoencoder.db import get_columns
from autoencoder.db import query_to_df

## Database schema

In [2]:
conn  = get_connection()
print("Connected to database")

Connected to database


In [3]:
for table in get_tables(conn):
    columns = get_columns(conn, table)
    col_str = ", ".join(f"{name} ({dtype})" for name, dtype in columns)
    print(f"  {table}: {col_str}")

  album_images: album_rowid (INTEGER), width (INTEGER), height (INTEGER), url (TEXT)
  albums: rowid (INTEGER), id (TEXT), fetched_at (INTEGER), name (TEXT), album_type (TEXT), available_markets_rowid (INTEGER), external_id_upc (TEXT), copyright_c (TEXT), copyright_p (TEXT), label (TEXT), popularity (INTEGER), release_date (TEXT), release_date_precision (TEXT), total_tracks (INTEGER), external_id_amgid (TEXT)
  artist_albums: artist_rowid (INTEGER), album_rowid (INTEGER), is_appears_on (INTEGER), is_implicit_appears_on (INTEGER), index_in_album (INTEGER)
  artist_genres: artist_rowid (INTEGER), genre (TEXT)
  artist_images: artist_rowid (INTEGER), width (INTEGER), height (INTEGER), url (TEXT)
  artists: rowid (INTEGER), id (TEXT), fetched_at (INTEGER), name (TEXT), followers_total (INTEGER), popularity (INTEGER)
  audio_features: track_rowid (INT), track_id (TEXT), time_signature (INT), tempo (INT), key (INT), mode (INT), danceability (REAL), energy (REAL), loudness (REAL), speechiness

In [18]:
tracks = query_to_df(
    conn,
    "SELECT "
    "    t.rowid as track_rowid, "
    "    t.name as track_name, "
    "    t.popularity as track_popularity, "
    "    t.duration_ms, af.time_signature "
    "FROM tracks AS t "
    "INNER JOIN audio_features AS af ON t.rowid = af.rowid "
    "WHERE t.popularity > 95"
)
tracks

,track_rowid,track_name,track_popularity,duration_ms,time_signature
0,290,That’s So True,96,166300,4
1,2779,Not Like Us,96,274192,1
2,2985,BAILE INoLVIDABLE,96,367725,4
3,87640805,Clean Baby Sleep White Noise (Loopable),97,142222,4
4,269,BIRDS OF A FEATHER,98,210373,0
5,2998,DtMF,98,237117,4
6,1178,Die With A Smile,100,251667,4


In [26]:
tracks = query_to_df(
    conn,
    """
    SELECT 
        t.rowid as track_rowid, 
        t.name as track_name, 
        a.name as artist_name 
    FROM tracks AS t 
    INNER JOIN track_artists AS ta ON t.rowid = ta.track_rowid 
    INNER JOIN artists AS a ON ta.artist_rowid = a.rowid 
    WHERE t.popularity > 60
      AND ta.artist_rowid = (
          SELECT artist_rowid
          FROM track_artists 
          WHERE track_rowid = t.rowid
          LIMIT 1 -- for the moment we limit to the track's first artist
      );
    """
)
tracks

,track_rowid,track_name,artist_name
0,245,Bar Named Jesus (ft. Thomas Rhett),Adam Doleac
1,284,Gave You I Gave You I,Gracie Abrams
2,285,Normal Thing,Gracie Abrams
3,347,Rose Colored Lenses,Miley Cyrus
4,404,Dance In The Dark,Lady Gaga
...,...,...,...
48096,2985,BAILE INoLVIDABLE,Bad Bunny
48097,87640805,Clean Baby Sleep White Noise (Loopable),Dream Supplier
48098,269,BIRDS OF A FEATHER,Billie Eilish
48099,2998,DtMF,Bad Bunny


In [25]:
tracks = query_to_df(
    conn,
    """
    SELECT 
        t.rowid as track_rowid, 
        t.name as track_name, 
        a.name as artist_name, 
        a.rowid as artist_rowid, 
        t.album_rowid, 
        al.name as album_name, 
        al.album_type, 
        al.label, 
        al.release_date, 
        al.release_date_precision, 
        t.external_id_isrc as id_isrc, 
        al.external_id_upc as id_upc, 
        af.time_signature, 
        af.tempo, 
        af.key, 
        af.mode, 
        af.danceability, 
        af.energy, 
        af.loudness, 
        af.speechiness, 
        af.acousticness, 
        af.instrumentalness, 
        af.liveness, 
        af.valence, 
        t.explicit, 
        t.popularity as track_popularity, 
        a.popularity as artist_popularity, 
        a.followers_total as followers, 
        al.popularity as album_popularity, 
        t.duration_ms, 
        t.track_number, 
        t.disc_number, 
        al.total_tracks
    FROM tracks AS t 
    INNER JOIN audio_features AS af ON t.rowid = af.track_rowid
    INNER JOIN albums AS al ON t.album_rowid = al.rowid 
    INNER JOIN track_artists AS ta ON t.rowid = ta.track_rowid -- joint table
    INNER JOIN artists AS a ON ta.artist_rowid = a.rowid 
    WHERE t.popularity > 60
      AND ta.artist_rowid = (
          SELECT artist_rowid
          FROM track_artists 
          WHERE track_rowid = t.rowid
          LIMIT 1 -- for the moment we limit to the track's first artist
      );
    """
)
tracks[tracks["album_name"] == "Watch The Throne"]

,track_rowid,track_name,artist_name,artist_rowid,album_rowid,album_name,album_type,label,release_date,release_date_precision,...,valence,explicit,track_popularity,artist_popularity,followers,album_popularity,duration_ms,track_number,disc_number,total_tracks
12307,10544,Gotta Have It,JAY-Z,3652986,1340,Watch The Throne,album,Roc Nation/RocAFella/IDJ,2011-08-08,day,...,0.330,1,63,84,9944087,75,140746,5,1,12
24292,51890436,Gotta Have It,JAY-Z,3652986,7253524,Watch The Throne,album,Roc Nation / Jay-Z,2011-08-08,day,...,0.330,1,65,84,9944087,55,140746,5,1,12
36225,10543,Otis,JAY-Z,3652986,1340,Watch The Throne,album,Roc Nation/RocAFella/IDJ,2011-08-08,day,...,0.432,1,70,84,9944087,75,178213,4,1,12
37997,10551,Why I Love You,JAY-Z,3652986,1340,Watch The Throne,album,Roc Nation/RocAFella/IDJ,2011-08-08,day,...,0.370,1,71,84,9944087,75,201453,12,1,12
43919,10540,No Church In The Wild,JAY-Z,3652986,1340,Watch The Throne,album,Roc Nation/RocAFella/IDJ,2011-08-08,day,...,0.557,1,76,84,9944087,75,272506,1,1,12
46409,10542,Ni**as In Paris,JAY-Z,3652986,1340,Watch The Throne,album,Roc Nation/RocAFella/IDJ,2011-08-08,day,...,0.775,1,81,84,9944087,75,219333,3,1,12
